# Lab 03 — Embeddings & Vector Distance Functions

Embeddings convert text into numerical vectors for similarity search. Vector distances measure how close two vectors are.

| Item | Detail |
|---|---|
| **Functions** | `AI_EMBED` (768-dim), `AI_EMBED` (1024-dim), `VECTOR_COSINE_SIMILARITY`, `VECTOR_INNER_PRODUCT`, `VECTOR_L1_DISTANCE`, `VECTOR_L2_DISTANCE` |
| **Exam Domain** | 2.0 Gen AI & LLM Functions |
| **You'll learn** | Embedding models, vector dimensions, all 4 distance/similarity metrics |


## Step 1 — Environment Setup

> **AI SQL Functions** — This lab uses the preferred `AI_` prefixed functions. 
Equivalent legacy functions: `AI_EMBED` (replaces `SNOWFLAKE.CORTEX.EMBED_TEXT_768 / EMBED_TEXT_1024`).

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;

CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — AI_EMBED (768-dim models) vs AI_EMBED (1024-dim models)

| Function | Dimensions | Trade-off |
|---|---|---|
| `AI_EMBED (768-dim models)` | 768 | Lighter, faster, lower storage cost |
| `AI_EMBED (1024-dim models)` | 1024 | Higher fidelity, better for complex semantic tasks |

Both use the `e5-base-v2` model and return a VECTOR type.


In [ ]:
SELECT
    'AI_EMBED (768-dim)' AS model,
    ARRAY_SIZE(AI_EMBED('e5-base-v2', 'Snowflake data warehouse')::ARRAY) AS dimensions
UNION ALL
SELECT
    'AI_EMBED (1024-dim)',
    ARRAY_SIZE(AI_EMBED('snowflake-arctic-embed-l-v2.0', 'Snowflake data warehouse')::ARRAY);

## Step 2b — Multilingual Embeddings

EMBED_TEXT_1024 supports multilingual models like `multilingual-e5-large` and `voyage-multilingual-2`. These produce semantically similar vectors for the same concept across languages.

| Model | Dimensions | Multilingual |
|---|---|---|
| `e5-base-v2` | 768 | English only |
| `snowflake-arctic-embed-m-v1.5` | 768 | English only |
| `snowflake-arctic-embed-l-v2.0` | 1024 | Multilingual |
| `multilingual-e5-large` | 1024 | 100+ languages |
| `voyage-multilingual-2` | 1024 | Multilingual |

In [ ]:
SELECT
    text_pair,
    ROUND(VECTOR_COSINE_SIMILARITY(
        AI_EMBED('snowflake-arctic-embed-l-v2.0', english),
        AI_EMBED('snowflake-arctic-embed-l-v2.0', other_lang)
    ), 4) AS cross_lingual_similarity
FROM (
    SELECT 'EN↔ES' AS text_pair,
        'cloud data warehouse' AS english,
        'almacén de datos en la nube' AS other_lang
    UNION ALL
    SELECT 'EN↔DE', 'machine learning model', 'maschinelles Lernmodell'
    UNION ALL
    SELECT 'EN↔FR', 'real-time analytics', 'analytique en temps réel'
);

## Step 2b — Multilingual Embeddings

EMBED_TEXT_1024 supports multilingual models like `multilingual-e5-large` and `voyage-multilingual-2`. These produce semantically similar vectors for the same concept across languages.

| Model | Dimensions | Multilingual |
|---|---|---|
| `e5-base-v2` | 768 | English only |
| `snowflake-arctic-embed-m-v1.5` | 768 | English only |
| `snowflake-arctic-embed-l-v2.0` | 1024 | Multilingual |
| `multilingual-e5-large` | 1024 | 100+ languages |
| `voyage-multilingual-2` | 1024 | Multilingual |

In [ ]:
SELECT
    text_pair,
    ROUND(VECTOR_COSINE_SIMILARITY(
        AI_EMBED('snowflake-arctic-embed-l-v2.0', english),
        AI_EMBED('snowflake-arctic-embed-l-v2.0', other_lang)
    ), 4) AS cross_lingual_similarity
FROM (
    SELECT 'EN↔ES' AS text_pair,
        'cloud data warehouse' AS english,
        'almacén de datos en la nube' AS other_lang
    UNION ALL
    SELECT 'EN↔DE', 'machine learning model', 'maschinelles Lernmodell'
    UNION ALL
    SELECT 'EN↔FR', 'real-time analytics', 'analytique en temps réel'
);

## Step 3 — All Vector Distance Functions

| Function | What It Measures | Range | Best For |
|---|---|---|---|
| `VECTOR_COSINE_SIMILARITY` | Angular similarity | -1 to 1 (1 = identical) | Text similarity, RAG |
| `VECTOR_INNER_PRODUCT` | Dot product | Unbounded (higher = more similar) | Ranking, recommendation |
| `VECTOR_L1_DISTANCE` | Manhattan distance | 0+ (lower = more similar) | Sparse features |
| `VECTOR_L2_DISTANCE` | Euclidean distance | 0+ (lower = more similar) | Geometric similarity |


In [ ]:
WITH embeddings AS (
    SELECT
        AI_EMBED('e5-base-v2', 'cloud data warehouse') AS vec_a,
        AI_EMBED('e5-base-v2', 'data storage platform') AS vec_b,
        AI_EMBED('e5-base-v2', 'chocolate cake recipe') AS vec_c
)
SELECT
    'warehouse vs platform (similar)' AS comparison,
    ROUND(VECTOR_COSINE_SIMILARITY(vec_a, vec_b), 4) AS cosine_sim,
    ROUND(VECTOR_INNER_PRODUCT(vec_a, vec_b), 4) AS inner_product,
    ROUND(VECTOR_L1_DISTANCE(vec_a, vec_b), 4) AS l1_distance,
    ROUND(VECTOR_L2_DISTANCE(vec_a, vec_b), 4) AS l2_distance
FROM embeddings
UNION ALL
SELECT
    'warehouse vs cake (dissimilar)',
    ROUND(VECTOR_COSINE_SIMILARITY(vec_a, vec_c), 4),
    ROUND(VECTOR_INNER_PRODUCT(vec_a, vec_c), 4),
    ROUND(VECTOR_L1_DISTANCE(vec_a, vec_c), 4),
    ROUND(VECTOR_L2_DISTANCE(vec_a, vec_c), 4)
FROM embeddings;

## Step 4 — Storing Embeddings in a Table

In practice, you compute embeddings once and store them for repeated search.

In [ ]:
CREATE OR REPLACE TABLE embedding_store (
    text_id INT AUTOINCREMENT,
    source_text TEXT,
    embedding VECTOR(FLOAT, 768)
);

INSERT INTO embedding_store (source_text, embedding)
SELECT column1, AI_EMBED('e5-base-v2', column1)
FROM VALUES
    ('Snowflake is a cloud data warehouse'),
    ('Machine learning models require training data'),
    ('SQL is used to query databases'),
    ('Cortex AI provides LLM functions in SQL'),
    ('Chocolate chip cookies are delicious');

SELECT text_id, source_text FROM embedding_store;

## Step 5 — Nearest-Neighbor Search

In [ ]:
SELECT
    source_text,
    ROUND(VECTOR_COSINE_SIMILARITY(
        embedding,
        AI_EMBED('e5-base-v2', 'How do I run queries on Snowflake?')
    ), 4) AS similarity
FROM embedding_store
ORDER BY similarity DESC;

## Key Takeaways

- `AI_EMBED` (768-dim) is lighter/faster; `AI_EMBED` (1024-dim) is higher fidelity.
- **Cosine similarity** is the go-to metric for text similarity and RAG.
- **Inner product** works well for ranking when vectors are normalized.
- **L1/L2 distances** measure geometric distance (lower = more similar).
- Store embeddings in VECTOR columns for efficient repeated search.
